In [0]:
%run 
"./00setup_config"

Connected. Database ready.


path,name,size,modificationTime
abfss://landing@medallionhealthdata03.dfs.core.windows.net/appointments.csv,appointments.csv,10907,1783938174000
abfss://landing@medallionhealthdata03.dfs.core.windows.net/billing.csv,billing.csv,10018,1783938174000
abfss://landing@medallionhealthdata03.dfs.core.windows.net/doctors.csv,doctors.csv,962,1783938174000
abfss://landing@medallionhealthdata03.dfs.core.windows.net/patients.csv,patients.csv,5626,1783938174000
abfss://landing@medallionhealthdata03.dfs.core.windows.net/treatments.csv,treatments.csv,11072,1783938174000


Batch ID : 1524e1b5-6873-4861-8dc1-232d95be04dd
Pipeline Version : 1.0
Run Time : 2026-07-13 11:30:21.352337


# Setting metadata

source_name,file_name,target_table,primary_key,load_type,active_flag
patients,patients.csv,bronze_patients,Patient_ID,FULL,Y
doctors,doctors.csv,bronze_doctors,Doctor_ID,FULL,Y
appointments,appointments.csv,bronze_appointments,Appointment_ID,FULL,Y
billing,billing.csv,bronze_billing,Billing_ID,FULL,Y
treatments,treatments.csv,bronze_treatments,Treatment_ID,FULL,Y


source_name,file_name,target_table,primary_key,load_type,active_flag
patients,patients.csv,bronze_patients,Patient_ID,FULL,Y
doctors,doctors.csv,bronze_doctors,Doctor_ID,FULL,Y
appointments,appointments.csv,bronze_appointments,Appointment_ID,FULL,Y
billing,billing.csv,bronze_billing,Billing_ID,FULL,Y
treatments,treatments.csv,bronze_treatments,Treatment_ID,FULL,Y


batch_id,source_name,layer,rows_read,rows_written,status,start_time,end_time,error_message


In [0]:
from pyspark.sql import functions as F

In [0]:
patients = spark.read.format("delta").load(f"{silver_path}patients")
appointments = spark.read.format("delta").load(f"{silver_path}appointments")
billing = spark.read.format("delta").load(f"{silver_path}billing")
doctors = spark.read.format("delta").load(f"{silver_path}doctors")
treatments = spark.read.format("delta").load(f"{silver_path}treatments")

patient360 = spark.read.format("delta").load(f"{gold_path}patient_360")
dashboard = spark.read.format("delta").load(f"{gold_path}executive_dashboard")

#Record count

In [0]:

counts = [

("Patients", patients.count()),
("Doctors", doctors.count()),
("Appointments", appointments.count()),
("Billing", billing.count()),
("Treatments", treatments.count()),
("Patient360", patient360.count())

]

count_df = spark.createDataFrame(
    counts,
    ["table_name","record_count"]
)

display(count_df)

table_name,record_count
Patients,50
Doctors,10
Appointments,200
Billing,200
Treatments,200
Patient360,200


#Null check

In [0]:
display(

patient360.select(

[
F.count(
F.when(F.col(c).isNull(),c)
).alias(c)

for c in patient360.columns

]

)

)
     

patient_id,First_Name,Last_Name,Gender,date_of_birth,insurance_provider,doctor_name,specialization,hospital_branch,years_experience,appointment_id,appointment_date,status,reason_for_visit,treatment_id,treatment_type,cost,bill_id,amount,payment_method,payment_status
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


#Duplicates check

In [0]:
duplicates = [

("Patients",

patients.count()-
patients.dropDuplicates(["patient_id"]).count()),

("Doctors",

doctors.count()-
doctors.dropDuplicates(["doctor_id"]).count()),

("Appointments",

appointments.count()-
appointments.dropDuplicates(["appointment_id"]).count()),

("Billing",

billing.count()-
billing.dropDuplicates(["bill_id"]).count()),

("Treatments",

treatments.count()-
treatments.dropDuplicates(["treatment_id"]).count())

]

duplicate_df = spark.createDataFrame(
duplicates,
["table","duplicate_records"]
)

display(duplicate_df)

table,duplicate_records
Patients,0
Doctors,0
Appointments,0
Billing,0
Treatments,0


#Revenue Validation

In [0]:
display(

patient360.agg(

F.sum("amount").alias("total_revenue"),

F.avg("amount").alias("average_bill"),

F.max("amount").alias("highest_bill"),

F.min("amount").alias("lowest_bill")

)

)

total_revenue,average_bill,highest_bill,lowest_bill
551249.85,2756.24925,4973.63,534.03


# Appointment Status

In [0]:
display(

patient360.groupBy("status")

.count()

.orderBy(F.desc("count"))

)

status,count
NO-SHOW,52
CANCELLED,51
SCHEDULED,51
COMPLETED,46


#Payment Status

In [0]:
display(

patient360.groupBy("payment_status")

.count()

.orderBy(F.desc("count"))

)

payment_status,count
PENDING,69
FAILED,67
PAID,64


#Storage Validation

In [0]:

display(dbutils.fs.ls(bronze_path))

display(dbutils.fs.ls(silver_path))

display(dbutils.fs.ls(gold_path))

path,name,size,modificationTime
abfss://bronze@medallionhealthdata03.dfs.core.windows.net/appointments/,appointments/,0,1783939020000
abfss://bronze@medallionhealthdata03.dfs.core.windows.net/audit_log/,audit_log/,0,1783938655000
abfss://bronze@medallionhealthdata03.dfs.core.windows.net/billing/,billing/,0,1783939025000
abfss://bronze@medallionhealthdata03.dfs.core.windows.net/doctors/,doctors/,0,1783939015000
abfss://bronze@medallionhealthdata03.dfs.core.windows.net/metadata_config/,metadata_config/,0,1783938588000
abfss://bronze@medallionhealthdata03.dfs.core.windows.net/patients/,patients/,0,1783938981000
abfss://bronze@medallionhealthdata03.dfs.core.windows.net/treatments/,treatments/,0,1783939029000


path,name,size,modificationTime
abfss://silver@medallionhealthdata03.dfs.core.windows.net/appointments/,appointments/,0,1783939810000
abfss://silver@medallionhealthdata03.dfs.core.windows.net/billing/,billing/,0,1783940176000
abfss://silver@medallionhealthdata03.dfs.core.windows.net/doctors/,doctors/,0,1783940800000
abfss://silver@medallionhealthdata03.dfs.core.windows.net/patients/,patients/,0,1783939355000
abfss://silver@medallionhealthdata03.dfs.core.windows.net/treatments/,treatments/,0,1783940403000


path,name,size,modificationTime
abfss://gold@medallionhealthdata03.dfs.core.windows.net/analytics_dataset/,analytics_dataset/,0,1783941444000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/appointment_summary/,appointment_summary/,0,1783941633000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/branch_performance/,branch_performance/,0,1783941770000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/doctor_performance/,doctor_performance/,0,1783941729000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/executive_dashboard/,executive_dashboard/,0,1783941957000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/kpi_billing_accuracy/,kpi_billing_accuracy/,0,1783941698000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/kpi_no_show/,kpi_no_show/,0,1783941658000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/kpi_summary/,kpi_summary/,0,1783941967000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/monthly_revenue/,monthly_revenue/,0,1783941568000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/operating_cost_per_patient/,operating_cost_per_patient/,0,1783941868000


#Pipeline Summary

In [0]:
summary = [
("Bronze Layer","Completed"),
("Silver Layer","Completed"),
("Gold Layer","Completed"),
("Patient360","Created"),
("Executive Dashboard","Created"),
("KPI Tables","Created")
]
summary_df = spark.createDataFrame(
summary,
["Pipeline Stage","Status"]
)
display(summary_df)

Pipeline Stage,Status
Bronze Layer,Completed
Silver Layer,Completed
Gold Layer,Completed
Patient360,Created
Executive Dashboard,Created
KPI Tables,Created


In [0]:

display(dashboard)

total_patients,total_doctors,total_appointments,completed_appointments,cancelled_appointments,no_show_appointments,total_revenue,average_bill,highest_bill,lowest_bill,average_treatment_cost,total_treatment_cost,top_doctor,top_branch,top_treatment,top_patient,most_used_payment,revenue_per_patient,revenue_per_doctor,billing_accuracy_pct,no_show_rate_pct
48,10,200,46,51,0,551249.85,2756.25,4973.63,534.03,2756.25,551249.85,Sarah Taylor,CENTRAL HOSPITAL,CHEMOTHERAPY,P012,CREDIT CARD,11484.37,55124.99,48.12,0.0


In [0]:
print("="*100)
print("Healthcare Medallion Architecture Pipeline Completed")
print("="*100)

print("Bronze Layer : SUCCESS")
print("Silver Layer : SUCCESS")
print("Gold Layer : SUCCESS")

print("="*100)

Healthcare Medallion Architecture Pipeline Completed
Bronze Layer : SUCCESS
Silver Layer : SUCCESS
Gold Layer : SUCCESS
